[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/4-memory.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 4. Session and durable memory

The message list dies when the chat ends. `data/customer_preference.md` does not.

Both sample customers use the same fields: Name, Allergy, Favorite, Last order, Notes. `get_preference` takes a name, not a filename. It returns that customer's entry only.

Maya starts a fresh session. The message list is empty except for this question. Her allergy and last order still come back. Jonah's question is a second fresh session, not a continuation of Maya's.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## The file

Two customers, one shape. The tool opens the heading. The model does not see the path as an argument.

In [ ]:
import tutorial.common.facts
import tutorial.common.memory  # registers get_preference

from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.memory import PREFERENCE_PATH, get_preference

print("preference file:", PREFERENCE_PATH)
print(get_preference({"name": "Maya"}))
print(get_preference({"name": "Jonah"}))

## Fresh sessions

Each `run_agent` call starts a new message list. Nothing Maya said is in Jonah's turn.

In [ ]:
print("=== fresh session, Maya ===")
maya = run_agent(
    "I'm Maya. This is a new chat. What is my allergy, and what did I last order?",
    system_text(),
)
print("ANSWER:", maya["text"])
check("get_preference" in "\n".join(maya["tool_log"]), "Maya's entry came from the file")
check("sesame" in maya["text"].lower(), "Maya's allergy is in the answer")
check("chili" in maya["text"].lower(), "Maya's last order is in the answer")

print("=== fresh session, Jonah ===")
jonah = run_agent(
    "I'm Jonah. New chat, nothing carried over. What did I last order?",
    system_text(),
)
print("ANSWER:", jonah["text"])
check("get_preference" in "\n".join(jonah["tool_log"]), "Jonah's entry came from the file")
check("gift" in jonah["text"].lower(), "Jonah's last order is the gift box")